# Uncertainty Analysis of Recovered Memory Kernel

This notebook demonstrates uncertainty quantification for the recovered memory function H(t).

Methods demonstrated:

1. Posterior covariance uncertainty
2. Confidence interval estimation
3. Bootstrap uncertainty analysis


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parents[0]
sys.path.append(str(ROOT))


## Import uncertainty modules


In [ ]:
from src.uncertainty.posterior_covariance import (
    compute_posterior_covariance,
    memory_confidence_interval
)

from src.uncertainty.bootstrap import (
    bootstrap_memory_recovery,
    bootstrap_statistics
)


## Load recovered memory kernel


In [ ]:
memory_file = ROOT / 'results' / 'recovered_memory.csv'

if memory_file.exists():
    data = pd.read_csv(memory_file)
    time = data['time'].values
    memory = data['recovered_memory'].values
else:
    time = np.linspace(0,100,200)
    memory = np.exp(-0.05*time)

plt.figure(figsize=(7,4))
plt.plot(time, memory)
plt.xlabel('Time')
plt.ylabel('Recovered H(t)')
plt.title('Recovered Memory Kernel')
plt.show()


## Posterior covariance uncertainty

For Bayesian inversion:

$$C_H=(K^T\Sigma^{-1}K+\lambda L^TL)^{-1}$$


In [ ]:
# Demonstration using a synthetic covariance matrix

n = len(memory)

covariance = np.diag(
    (0.05 * np.maximum(memory, 1e-6))**2
)

lower, upper = memory_confidence_interval(
    memory,
    covariance,
    confidence=0.95
)

plt.figure(figsize=(7,4))
plt.plot(time, memory, label='H(t)')
plt.fill_between(
    time,
    lower,
    upper,
    alpha=0.3,
    label='95% interval'
)
plt.xlabel('Time')
plt.ylabel('Memory')
plt.legend()
plt.show()


## Bootstrap uncertainty

Bootstrap analysis estimates variability by repeated perturbation and recovery.

In [ ]:
def example_recovery(signal):
    return signal

samples = bootstrap_memory_recovery(
    memory,
    example_recovery,
    n_bootstrap=100,
    noise_level=0.03,
    random_seed=42
)

bootstrap_mean, bootstrap_std = bootstrap_statistics(samples)

plt.figure(figsize=(7,4))
plt.plot(time, bootstrap_mean, label='Bootstrap mean')
plt.fill_between(
    time,
    bootstrap_mean-bootstrap_std,
    bootstrap_mean+bootstrap_std,
    alpha=0.3,
    label='±1 standard deviation'
)
plt.xlabel('Time')
plt.ylabel('Memory')
plt.legend()
plt.show()
